In [ ]:
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np
from skimage.io import imread
from scipy.interpolate import RegularGridInterpolator

from edt import edt
from nd2 import ND2File


def get_pixel_size(file_path):
    with ND2File(file_path) as reader:
        # invert xyz voxel size to zyx to match img array
        pixel_size = reader.voxel_size()[::-1]
    return pixel_size


def get_edts(mask, pixel_size):
    # do 3D EDT
    dt = edt(mask, anisotropy=pixel_size)

    # get size of whole image in units
    img_size = np.array(pixel_size) * np.array(mask.shape)

    # do xy-EDT for each plane in stack
    # NOTE: to avoid doing this in a loop, we just set a fake pixel size in z
    # corresponding to the image size in xy (larger of the 2), thus all paths leading to a pixel should be just in xy 
    fake_pixel_size_xy_dt = [img_size[1:].max()] + list(pixel_size[1:])
    dt_xy = edt(mask, anisotropy=fake_pixel_size_xy_dt)

    # get distance just in z via fake xy pixel sizes corresponding to maximal z extent (see above)
    fake_pixel_size_z_dt = [pixel_size[0], img_size[0], img_size[0]]
    dt_z = edt(mask, anisotropy=fake_pixel_size_z_dt)

    return dt, dt_xy, dt_z


def get_distance_table(mask, spot_df, pixel_size, coord_column_names=['z', 'y', 'x']):

    # calculate EDTs
    dt, dt_xy, dt_z = get_edts(mask, pixel_size)

    # make interpolators so we can access distances at subpixel locations
    # EDTs: default linear interpolation
    interp_dt = RegularGridInterpolator(tuple((np.arange(s) for s in dt.shape)), dt, bounds_error=False, fill_value=None)
    interp_dt_xy = RegularGridInterpolator(tuple((np.arange(s) for s in dt_xy.shape)), dt_xy, bounds_error=False, fill_value=None)
    interp_dt_z = RegularGridInterpolator(tuple((np.arange(s) for s in dt_z.shape)), dt_z, bounds_error=False, fill_value=None)
    # mask interpolator: use nearest neighbor
    interp_mask = RegularGridInterpolator(tuple((np.arange(s) for s in mask.shape)), mask, method='nearest', bounds_error=False, fill_value=None)
    
    # "chache" of sorted distances for each label
    dt_sorted = {}
    dt_sorted_xy = {}
    dt_sorted_z = {}

    # dict of lists to build DataFrame
    res_df = defaultdict(list)

    for _, coords in spot_df[coord_column_names].iterrows():

        # get label from mask (nearest-neighbor interpolated)
        label = interp_mask(coords).astype(mask.dtype).item()

        # sort all distances of object with index label (except background) and store in cache
        if label != 0 and not label in dt_sorted:
            dt_sorted[label] = np.sort(dt[mask==label])
            dt_sorted_xy[label] = np.sort(dt_xy[mask==label])
            dt_sorted_z[label] = np.sort(dt_z[mask==label])

        # get distances from interpolated EDTs
        d = interp_dt(coords).item()
        d_xy = interp_dt_xy(coords).item()
        d_z = interp_dt_z(coords).item()

        # get quantiles in EDTs (index at which d would be inserted / total size)
        q = np.searchsorted(dt_sorted[label], d) / dt_sorted[label].size if label != 0 else np.nan
        q_xy = np.searchsorted(dt_sorted_xy[label], d_xy) / dt_sorted_xy[label].size if label != 0 else np.nan
        q_z = np.searchsorted(dt_sorted_z[label], d_z) / dt_sorted_z[label].size if label != 0 else np.nan

        res_df['label'].append(label)

        res_df['d'].append(d)
        res_df['d_xy'].append(d_xy)
        res_df['d_z'].append(d_z)

        res_df['q'].append(q)
        res_df['q_xy'].append(q_xy)
        res_df['q_z'].append(q_z)

    res_df = pd.DataFrame(res_df)
    return res_df

In [ ]:
in_path = '/data/agl_data/NanoFISH/Gabi/GS075_20230818_K562-EVI1-GFP_t(3-8)_EVI-CTRL/'

mask_subdirectory = 'segmentation_nuclei1_edgesnap'
spot_subdirectory = 'spot-detection-chromatic-shift-corrected'
out_subdirectory = 'spot-mask-distances'

# column names of (pixel) zyx coordinates in the spot detection table
coord_column_names = ['z_shift_corrected', 'y_shift_corrected', 'x_shift_corrected']

discard_spots_outside_mask = True


In [ ]:
base_path = Path(in_path)

mask_path = base_path / mask_subdirectory

spot_detection_path = base_path / spot_subdirectory

spot_feature_path = base_path / out_subdirectory

In [ ]:
mask_files = sorted(mask_path.glob('*.tif'))

spot_files = sorted(spot_detection_path.glob('*.csv'))

# check pairs of files
list(zip(mask_files, spot_files))

In [ ]:
# make out path if necessary
if not spot_feature_path.exists():
    spot_feature_path.mkdir(parents=True)


for mask_file, spot_file in zip(mask_files, spot_files):

    # load spot detection results
    spot_df = pd.read_csv(spot_file)

    # load pixel size from raw file
    # NOTE: column 'image_file' should be present in spot detection result table
    # if table is empty, default to [1, 1, 1] pixel size -> will not be used anyway
    pixel_size = get_pixel_size(spot_df['image_file'].iloc[0]) if len(spot_df) > 0 else [1, 1, 1]

    # load mask
    mask = imread(mask_file)

    res_df = get_distance_table(mask, spot_df, pixel_size, coord_column_names)
    res_df['spot_detection_file'] = spot_file
    res_df['mask_file'] = mask_file

    # add some info from original spot detection table as extra column 
    # -> this way, we can join later
    res_df['spot_idx'] = spot_df.spot_idx
    res_df['channel'] = spot_df['channel']
    res_df['image_file'] = spot_df['image_file']
    res_df['position_idx'] = spot_df['position_idx']

    if discard_spots_outside_mask and len(res_df) > 0:
        res_df = (res_df[res_df.label != 0])

    out_file_path = spot_feature_path / (spot_file.stem + '_spot-mask-distances.csv')
    res_df.to_csv(out_file_path, index=None)

    print(f'finished processing {mask_file}.')